In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import random
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, classification_report

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    device_name = 'mps'
else:
    device = torch.device('cpu')
    device_name = 'cpu'

print({'selected_device': device_name, 'seed': seed})


In [ ]:
dataset = load_dataset('dair-ai/emotion')
train_dataset = dataset['train']
test_dataset = dataset['test']
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

print({
    'train_rows': len(train_dataset),
    'test_rows': len(test_dataset),
    'num_classes': len(class_names)
})
print({'train_sample': train_dataset[:3], 'test_sample': test_dataset[:3]})


In [ ]:
examples_per_class = 8
subset_indices = []
for label_id in range(len(class_names)):
    label_indices = [i for i, y in enumerate(train_dataset['label']) if y == label_id]
    subset_indices.extend(label_indices[:examples_per_class])

train_subset = train_dataset.select(subset_indices)
train_texts = train_subset['text']
train_labels = np.array(train_subset['label'])

print({
    'examples_per_class': examples_per_class,
    'train_subset_size': len(train_subset),
    'class_counts': {class_names[i]: int((train_labels == i).sum()) for i in range(len(class_names))}
})
print(pd.DataFrame({
    'text': train_texts[:12],
    'label': [class_names[i] for i in train_labels[:12]]
}).to_dict(orient='records'))


In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print({'model_name': model_name, 'hidden_size': int(model.config.hidden_size)})


In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            model_output = model(**encoded)
            sentence_embeddings = mean_pooling(model_output, encoded['attention_mask'])
            sentence_embeddings = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)
        all_embeddings.append(sentence_embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

train_embeddings = encode_texts(train_texts, batch_size=64, max_length=128)
print({'train_embedding_shape': list(train_embeddings.shape)})


In [ ]:
test_texts = test_dataset['text']
true_ids = np.array(test_dataset['label'])

test_embeddings = encode_texts(test_texts, batch_size=64, max_length=128)
similarity = test_embeddings @ train_embeddings.T
nn_indices = similarity.argmax(dim=1).numpy()
pred_ids = train_labels[nn_indices]

results_df = pd.DataFrame({
    'text': test_texts,
    'true_label': [class_names[i] for i in true_ids],
    'predicted_label': [class_names[i] for i in pred_ids],
    'nearest_train_text': [train_texts[i] for i in nn_indices],
    'nearest_train_label': [class_names[train_labels[i]] for i in nn_indices],
    'max_similarity': similarity.max(dim=1).values.numpy()
})

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'dair-ai/emotion',
    'train_subset_size': len(train_subset),
    'test_size': len(test_dataset),
    'retrieval_k': 1,
    'device': device_name,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 8
print(results_df.head(sample_n).to_string(index=False))
